In [10]:
import numpy as np
import pandas as pd
import matplotlib.pylab as plt
import torch 
from torch import nn, optim
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import sklearn
from sklearn.model_selection import train_test_split
import copy
import mytools
from pathlib import Path
import joblib


# Print and store device being used
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [ ]:
data_dir = '/Users/mghrear/data/HPS_data/2021_v9_pass5_processed/'
out_dir = '/Users/mghrear/data/HPS_data/2021_v9_pass5_preds_11/'

# /Users/mghrear/data/HPS_data/2021_v9_pass5_preds/ has ANN run 5 ANN sideband a run 1 ANN sideband b run 1
#/Users/mghrear/data/HPS_data/2021_v9_pass5_preds_2/ has ANN run 1 ANN sideband a run 1 (but its different) ANN sideband b run 1



#/Users/mghrear/data/HPS_data/2021_v9_pass5_preds_10/ has ANN_full run 1
#/Users/mghrear/data/HPS_data/2021_v9_pass5_preds_11/ has ANN_full run 2


for p in Path(data_dir).iterdir():

    # Load dataframe
    df = pd.read_pickle(data_dir+p.name)

    # Get InvM
    InvM = mytools.get_InvM(df) 

    # Setup dataloader
    test_dataset = TensorDataset(torch.from_numpy(df.to_numpy().astype(np.float32)))
    test_loader = DataLoader(test_dataset, batch_size=2000, shuffle=False)

    # Initialize ANN classifier
    ANN = mytools.Classifier(in_features=df.shape[1]).to(device)
    #ANN.load_state_dict(torch.load("/Users/mghrear/data/ML_data/patch/classifier_adv_2021_v9_pass5_run"+str(7)+"_limited.pt", map_location=device))
    ANN.load_state_dict(torch.load("/Users/mghrear/data/ML_data/patch/classifier_adv_2021_v9_pass5_run"+str(2)+".pt", map_location=device))
    ANN.eval()
    # Initialize ANN classifier trained with sidebands method a
    ANN_sideband_a = mytools.Classifier(in_features=df.shape[1]).to(device)
    ANN_sideband_a.load_state_dict(torch.load("/Users/mghrear/data/ML_data/patch/classifier_adv_2021_v9_pass5_run"+str(1)+"_limited_sideband_a.pt", map_location=device))
    ANN_sideband_a.eval() 
    # Initialize ANN classifier trained with sidebands method b
    ANN_sideband_b = mytools.Classifier(in_features=df.shape[1]).to(device)
    ANN_sideband_b.load_state_dict(torch.load("/Users/mghrear/data/ML_data/patch/classifier_adv_2021_v9_pass5_run"+str(1)+"_limited_sideband_b.pt", map_location=device))
    ANN_sideband_b.eval()

    # Initialize BDT classifier
    bdt_model = joblib.load('/Users/mghrear/data/ML_data/patch/BDT_2021_v9_pass5_limited_model.pkl')
    # Initialize BDT classifier trained with sidebands method a
    bdt_model_sideband_a = joblib.load('/Users/mghrear/data/ML_data/patch/BDT_2021_v9_pass5_limited_sideband_a_model.pkl')
    # Initialize BDT classifier trained with sidebands method b
    bdt_model_sideband_b = joblib.load('/Users/mghrear/data/ML_data/patch/BDT_2021_v9_pass5_limited_sideband_b_model.pkl')


    # Get predicitions
    ANN_pred = mytools.test_clas(test_loader, ANN, device)
    ANN_pred = torch.sigmoid(torch.tensor(ANN_pred)).numpy()
    ANN_pred_sideband_a = mytools.test_clas(test_loader, ANN_sideband_a, device)
    ANN_pred_sideband_a = torch.sigmoid(torch.tensor(ANN_pred_sideband_a)).numpy()
    ANN_pred_sideband_b = mytools.test_clas(test_loader, ANN_sideband_b, device)
    ANN_pred_sideband_b = torch.sigmoid(torch.tensor(ANN_pred_sideband_b)).numpy()
    bdt_model_pred = bdt_model.predict_proba(df)[:,1]
    bdt_model_pred_sideband_a = bdt_model_sideband_a.predict_proba(df)[:,1]
    bdt_model_pred_sideband_b = bdt_model_sideband_b.predict_proba(df)[:,1]

    # Add results to datafram
    df['ANN_pred'] = ANN_pred
    df['ANN_pred_sideband_a'] = ANN_pred_sideband_a
    df['ANN_pred_sideband_b'] = ANN_pred_sideband_b
    df['bdt_model_pred'] = bdt_model_pred
    df['bdt_model_pred_sideband_a'] = bdt_model_pred_sideband_a
    df['bdt_model_pred_sideband_b'] = bdt_model_pred_sideband_b

    df.to_pickle(out_dir+p.name)

